In [1]:
!pip uninstall brotli -y

Found existing installation: Brotli 1.0.9
Uninstalling Brotli-1.0.9:


ERROR: Exception:
Traceback (most recent call last):
  File "D:\Coding\anaconda\Lib\shutil.py", line 856, in move
    os.rename(src, real_dst)
    ~~~~~~~~~^^^^^^^^^^^^^^^
OSError: [WinError 17] The system cannot move the file to a different disk drive: 'd:\\coding\\anaconda\\lib\\site-packages\\_brotli.cp313-win_amd64.pyd' -> 'C:\\Users\\usama\\AppData\\Local\\Temp\\pip-uninstall-tjphoz1d\\_brotli.cp313-win_amd64.pyd'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "D:\Coding\anaconda\Lib\site-packages\pip\_internal\cli\base_command.py", line 105, in _run_wrapper
    status = _inner_run()
  File "D:\Coding\anaconda\Lib\site-packages\pip\_internal\cli\base_command.py", line 96, in _inner_run
    return self.run(options, args)
           ~~~~~~~~^^^^^^^^^^^^^^^
  File "D:\Coding\anaconda\Lib\site-packages\pip\_internal\commands\uninstall.py", line 106, in run
    uninstall_pathset = req.uninstall(
        auto_confirm=optio

In [11]:
!pip cache purge

Files removed: 426 (241.3 MB)


In [28]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_file = Path.cwd() / "secrets.env"
load_dotenv(env_file)

groq_key = os.getenv("GROQ_API_KEY")
openai_key = os.getenv("OPENAI_API_KEY")

if not groq_key:
    raise RuntimeError("GROQ_API_KEY is missing. Add it to secrets.env, then restart and run this cell.")

In [29]:
from openai import OpenAI
import httpx2

In [30]:
from langchain_community.document_loaders import TextLoader, WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_groq import ChatGroq

In [31]:
http_client = httpx2.Client(headers={"Accept-Encoding": "gzip, deflate"})
client = OpenAI(
    api_key= groq_key,
    base_url= 'https://api.groq.com/openai/v1/',
    http_client=http_client
)

In [32]:
def check_emotion(prompt):
    response = client.chat.completions.create(
        model = 'openai/gpt-oss-20b',
        messages= [
            {
                'role' : 'system',
                'content': f"you'll be given a sentence , check it's emotion"
            },
            {
                'role' : 'user',
                'content': "I'm so lonely"
            },
            {
                'role' : 'assistant',
                'content' : 'sad'
            },
            {
                'role' : 'user',
                'content': prompt
            }
            
        ]
    )
    return response.choices[0].message.content.strip()


In [33]:
print(check_emotion('funny world we live in '))

amused


In [7]:
!jupyter --version

'jupyter' is not recognized as an internal or external command,
operable program or batch file.


In [34]:
def sarcasm(prompt):
    response =  client.chat.completions.create(
        model = "openai/gpt-oss-20b",
        messages = [
            {
              "role" : "system",
                "content" : "you will be given a sentence , respond in a sarcastic manner"
            },
            {
                "role": "user",
                "content": "I only had one slice of pizza."
            },
            {
                "role": "assistant",
                "content": "Sure, and I only had one glass of wine at that wedding."
            },
            {
                "role": "user",
                "content": "I'm not mad."
            },
            {
                "role": "assistant",
                "content": "No, you're just aggressively fine, got it."
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    )
    
    return response.choices[0].message.content.strip()

print(sarcasm("today weather is very hot"))

Wow, because we all really need another reason to skip the ice cream truck, right?


In [35]:
source_url = 'https://en.wikipedia.org/wiki/Grand_Theft_Auto_VI'

In [36]:
webloader = WebBaseLoader(web_path= (source_url,))
raw_document = webloader.load()

In [37]:
textsplitter = RecursiveCharacterTextSplitter()
splitted_documents = textsplitter.split_documents(documents= raw_document)

In [38]:
embeddings = HuggingFaceEmbeddings(
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [39]:
vector_store = FAISS.from_documents(
    documents= splitted_documents,
    embedding= embeddings
)

In [40]:
memory = ConversationBufferMemory(
    memory_key= 'chat_history' , return_messages= True
)

In [44]:
qa = ConversationalRetrievalChain.from_llm(
    ChatGroq(
        model="openai/gpt-oss-20b",
        api_key= groq_key,
        temperature= 0.1,
    ),
    vector_store.as_retriever(),
    memory= memory

)

In [48]:
query = "Is dan houser still working on GTA 6 ?"

In [49]:
result =  qa.invoke({"question": query})

In [50]:
result['answer'].strip()

'No. Dan\u202fHouser is no longer involved in the development of Grand\u202fTheft\u202fAuto\u202f6.  In a September\u202f2025 interview, Rockstar co‑founder Dan\u202fHouser said that the game “is not going to be a story I wrote” and that he was not writing it, indicating that he had stepped away from the project (McCrae, *GamesRadar+*, 29\u202fSeptember\u202f2025).  Houser left Rockstar Games in 2017, so he is not part of the current GTA\u202f6 team.'

In [15]:
text_loader = TextLoader(
    file_path= 'Games record .txt',
    encoding= 'utf-8'
)
raw_document = text_loader.load()

In [16]:
text_splitter = RecursiveCharacterTextSplitter()
splitted_docs = text_splitter.split_documents(raw_document)

In [17]:
embeddings = HuggingFaceEmbeddings(
     model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [18]:
vector_store = FAISS.from_documents(documents= splitted_docs, embedding= embeddings)

In [19]:
memory = ConversationBufferMemory(memory_key= "chat_history" , return_messages= True)

In [20]:
qa = ConversationalRetrievalChain.from_llm(
    ChatGroq(
        model= 'openai/gpt-oss-20b',
        api_key= groq_key,
        temperature= 0.7
    ),
    vector_store.as_retriever(),
    memory = memory
)

In [21]:
 query = 'any game with rim written alongside it ?'

In [22]:
print("qa.memory:", qa.memory)
print("memory key:", qa.memory.memory_key if qa.memory else None)
print(qa.prep_inputs({"question": "test"}))

qa.memory: chat_memory=InMemoryChatMessageHistory(messages=[]) return_messages=True memory_key='chat_history'
memory key: chat_history
{'question': 'test', 'chat_history': []}


In [23]:
result = qa.invoke({
    "question": query,
})

In [24]:
result['answer'].strip()

'From the list you provided, the only title that has “rim” written alongside it is:\n\n- **Darkness** – listed as “Darkness (5‑12‑22)(rim)”\n\nNo other game in the list carries a “rim” annotation.'